# SmolVLA 全量微调 · SO-101 仿真抓取

**这个 notebook 在做什么**：把社区预训练的 SmolVLA（`lerobot/smolvla_base`，SmolVLM2-500M 底座 + flow-matching 动作专家）在 SO-101 仿真数据上做**全参数微调**，再回到仿真里评测成功率。它合上了一整条闭环：**仿真环境 → RL 训专家 → 专家 rollout 生成数据集 → VLA 全量微调 → 仿真评测**，全程不碰真机。

**与前后模块的关系**：数据集来自 `rl/3_offpolicy/3_2_so101_offpolicy`（RL 专家产的演示）；加载/推理接口和 4_4 的 SmolVLA 推理完全一致；本讲把视角从「用别人微调好的权重」推进到「自己微调一个权重」。讲12 3.9 节会把同一条命令换成 LoRA 微调，和这里的全量微调形成对照。

**看点**：① 微调只是一条 `lerobot-train` 命令，难点不在训练循环而在**特征对齐**（`rename_map`）；② 全量微调=更新全部可训练参数，没有 LoRA/PEFT 适配器；③ 训练用的 ReachCube 任务已经下线，评测只能落在现存的 PickPlaceCube40 上，所以这一版跑出来的 `pc_success≈0` 是**任务不匹配**、不是微调失败——第 3 节讲怎么读这个数。

> 在 `code/` 目录启动内核运行；训练/评测本体走同目录的 `train_smolvla.sh` / `smolvla_eval.sh`，本 notebook 负责讲机制、看数据、读结果。

## 1 数据从哪来：先看一眼数据集

微调数据是一个标准 `LeRobotDataset`，由 `rl/3_2` 的 `datagen/gen_dataset.py` 产出：SAC 专家在 `SO101ReachCube-v1` 上 rollout，再转成 LeRobot 格式。先直接读它的 `meta/info.json`，确认三件事——**相机叫什么名、几维状态/动作、多少集**。相机名这一条是后面 `rename_map` 的关键。

In [ ]:
import os
import json
from pathlib import Path

# 数据集落通用数据根下的生成目录（由 rl/3_2 的 datagen 写入）。
dataset_root = Path(os.environ["DATASETS_ROOT"]) / "so101_sim" / "_gen" / "SO101ReachCube-v1" / "dataset"
info = json.loads((dataset_root / "meta" / "info.json").read_text())

print("集数:", info["total_episodes"], "| 帧数:", info["total_frames"], "| fps:", info["fps"])
for key, feat in info["features"].items():
    print(f"  {key:34s} shape={feat['shape']} dtype={feat['dtype']}")
# 注意相机键是 observation.images.base_camera —— 单目一路，这与预训练权重的 camera1/2/3 不一致。

## 2 全量微调：一条命令 + 一个 `rename_map`

微调本体就是 `lerobot-train --policy.path=lerobot/smolvla_base`：它把预训练权重整体载入，用我们的数据集全参数更新（`use_peft=false`，没有 LoRA 适配器，这正是「全量」与 LoRA 微调的区别）。

**唯一的坑是相机名对不上**。`smolvla_base` 是按三路相机 `camera1/2/3` 预训练的，我们的数据集只有一路 `base_camera`。不处理会直接报：

```
ValueError: Feature mismatch ...
- Missing features: ['observation.images.camera1', ...]
- Extra features: ['observation.images.base_camera']
```

修法是把数据集的相机键**改名映射**到权重期望的键：

```bash
--rename_map='{"observation.images.base_camera": "observation.images.camera1"}'
```

缺 `camera2/3` 是允许的——特征校验只要求「数据提供的相机」是「权重期望相机」的**子集**。完整命令见 `train_smolvla.sh`：

```bash
lerobot-train \
    --policy.path=lerobot/smolvla_base --policy.push_to_hub=false \
    --dataset.root=$DATASETS_ROOT/so101_sim/_gen/SO101ReachCube-v1/dataset \
    --rename_map='{"observation.images.base_camera": "observation.images.camera1"}' \
    --batch_size=64 --steps=20000 --wandb.enable=false \
    --output_dir=$DATASETS_ROOT/models/trained/so101_sim_smolvla/SO101ReachCube-v1
```

训练时每 200 步打印一次 flow-matching 损失。下面把日志里的 loss 画成曲线，看它是否收敛。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# loss 曲线由 plot_loss.py 从训练日志生成；没跑过 train_smolvla.sh + plot_loss.py 的话，
# result/ 还是空的，下面这格会找不到文件。
here = Path.cwd() / "vla" / "5_vla_finetune" / "5_2_smolvla_full_sft"
plt.figure(figsize=(8, 4.5))
plt.imshow(mpimg.imread(here / "result" / "loss_curve.png"))
plt.axis("off")
plt.show()

## 3 仿真评测：pc_success 是硬指标

训练完在同一个仿真里评测。`lerobot-eval --env.type=so101_sim` 自带向量环境、多 episode 统计和自动录像；`so101_sim` 环境由 `code/platform/lerobot` 的 0004 补丁注册。

评测端同样要 `rename_map`：环境输出的观测键是 `base_camera`，而 checkpoint 是按 `camera1` 训练的，得把两者对齐。命令见 `smolvla_eval.sh`：

```bash
lerobot-eval \
    --policy.path=<output_dir>/checkpoints/last/pretrained_model \
    --rename_map='{"observation.images.base_camera": "observation.images.camera1"}' \
    --env.type=so101_sim --env.task=SO101PickPlaceCube40-v1 \
    --eval.n_episodes=20 --eval.batch_size=10
```

注意 `--env.task` 只能填 `so101_sim` 现存的场景（`SO101ReachCube-v1` 已下线），而这个 checkpoint 是在 ReachCube 数据上训的——评测环境和训练数据不是同一个任务，`pc_success≈0` 只说明任务不匹配，不代表微调本身失败。关键指标读法仍是 `pc_success`（成功率百分比）：

In [ ]:
eval_info = json.loads((here / "result" / "eval_smolvla" / "eval_info.json").read_text())
agg = eval_info["aggregated"]
print("pc_success:", agg["pc_success"], "%")
print("avg_sum_reward:", round(agg["avg_sum_reward"], 3))
print("评测 episode 数:", len(eval_info["per_episode"]))

## 4 小结

**数据流**：`smolvla_base` 预训练权重 + SO-101 仿真演示数据 → `lerobot-train` 全参数更新（`rename_map` 对齐相机名）→ 微调后的 checkpoint → `lerobot-eval` 在同一仿真里报 `pc_success`。

**一句话 takeaway**：VLA 微调工程上「跑起来」很简单（一条命令），真正卡人的是**观测特征对齐**——数据集的相机/状态命名必须映射到预训练权重期望的命名，`rename_map` 是这条闭环里最容易被忽略、又必须做对的一步。

**承上启下**：这里做的是全量微调；当模型更大（π0/π0.5）单卡放不下、或想省显存快速适配时，就在同一条 `lerobot-train` 后面加上 `--policy.use_peft=true --peft.method_type=LORA --peft.r=16` 换成 LoRA 微调——冻结主干、只训低秩适配器，命令结构和这里几乎一样（讲12 3.9 节）。